In [ ]:
#pip install open-rppg
import rppg
import time
import cv2

model = rppg.Model('ME-chunk.rlap')

with model.video_capture(0):
    last_process_time = 0
    current_hr = None
    current_sqi = None
    current_hrv = None

    for frame, box in model.preview:
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        h, w = frame.shape[:2]

        # 1초마다 HR + HRV 갱신
        now = time.time()
        if now - last_process_time > 1.0:
            # return_hrv=True가 기본값이므로 생략 가능
            result = model.hr(start=-30)  # HRV 계산은 30초 이상 권장
            if result and result['hr']:
                current_hr  = result['hr']
                current_sqi = result.get('SQI')
                current_hrv = result.get('hrv')
                
                # 콘솔 출력
                print(f"HR: {current_hr:.1f} BPM | SQI: {current_sqi:.2f}")
                if current_hrv:
                    print(
                        f"  SDNN: {current_hrv.get('sdnn', float('nan')):.1f} ms | "
                        f"RMSSD: {current_hrv.get('rmssd', float('nan')):.1f} ms | "
                        f"pNN50: {current_hrv.get('pnn50', float('nan')):.1f}% | "
                        f"LF/HF: {current_hrv.get('LF/HF', float('nan')):.2f} | "
                        f"BR: {current_hrv.get('breathingrate', float('nan')):.1f} /min"
                    )
            last_process_time = now

        # 2. 얼굴 박스 그리기
        if box is not None:
            y1, y2 = box[0]
            x1, x2 = box[1]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # 3. 화면 HUD 표시
        panel_x = 10
        line_h  = 28  # 줄 간격

        def put_label(img, text, row, color=(255, 255, 255)):
            cv2.putText(img, text, (panel_x, 30 + row * line_h),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 4, cv2.LINE_AA)  # 검정 외곽선
            cv2.putText(img, text, (panel_x, 30 + row * line_h),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, color,  2, cv2.LINE_AA)

        # HR & SQI
        hr_text  = f"HR : {current_hr:.1f} BPM"  if current_hr  else "HR : --"
        sqi_text = f"SQI: {current_sqi:.2f}"      if current_sqi else "SQI: --"
        sqi_color = (0, 255, 0) if (current_sqi or 0) >= 0.6 else (0, 165, 255)  # 녹색 / 주황

        put_label(frame, hr_text,  row=0, color=(0, 255, 128))
        put_label(frame, sqi_text, row=1, color=sqi_color)

        # HRV 지표
        if current_hrv:
            sdnn  = current_hrv.get('sdnn',          float('nan'))
            rmssd = current_hrv.get('rmssd',         float('nan'))
            pnn50 = current_hrv.get('pnn50',         float('nan'))
            lfhf  = current_hrv.get('LF/HF',         float('nan'))
            br    = current_hrv.get('breathingrate',  float('nan'))

            put_label(frame, f"SDNN : {sdnn:.1f} ms",   row=3, color=(200, 220, 255))
            put_label(frame, f"RMSSD: {rmssd:.1f} ms",  row=4, color=(200, 220, 255))
            put_label(frame, f"pNN50: {pnn50:.1f} %",   row=5, color=(200, 220, 255))
            put_label(frame, f"LF/HF: {lfhf:.2f}",      row=6, color=(200, 220, 255))
            put_label(frame, f"BR   : {br:.1f} /min",   row=7, color=(200, 220, 255))
        else:
            put_label(frame, "HRV : 데이터 수집 중...", row=3, color=(150, 150, 150))

        cv2.imshow("rPPG Monitor", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

HR: 78.4 BPM | SQI: 0.72
  SDNN: 43.3 ms | RMSSD: 40.2 ms | pNN50: 0.4% | LF/HF: 0.87 | BR: 0.2 /min
HR: 78.4 BPM | SQI: 0.72
  SDNN: 43.3 ms | RMSSD: 40.2 ms | pNN50: 0.4% | LF/HF: 0.87 | BR: 0.2 /min
HR: 78.4 BPM | SQI: 0.72
  SDNN: 43.3 ms | RMSSD: 40.2 ms | pNN50: 0.4% | LF/HF: 0.87 | BR: 0.2 /min
HR: 78.4 BPM | SQI: 0.72
  SDNN: 43.3 ms | RMSSD: 40.2 ms | pNN50: 0.4% | LF/HF: 0.87 | BR: 0.2 /min
HR: 78.4 BPM | SQI: 0.72
  SDNN: 43.3 ms | RMSSD: 40.2 ms | pNN50: 0.4% | LF/HF: 0.87 | BR: 0.2 /min
HR: 78.1 BPM | SQI: 0.84
  SDNN: 39.1 ms | RMSSD: 46.3 ms | pNN50: 0.4% | LF/HF: 0.07 | BR: 0.2 /min
HR: 78.1 BPM | SQI: 0.84
  SDNN: 39.1 ms | RMSSD: 46.3 ms | pNN50: 0.4% | LF/HF: 0.07 | BR: 0.2 /min
HR: 78.1 BPM | SQI: 0.84
  SDNN: 39.1 ms | RMSSD: 46.3 ms | pNN50: 0.4% | LF/HF: 0.07 | BR: 0.2 /min
HR: 78.1 BPM | SQI: 0.84
  SDNN: 39.1 ms | RMSSD: 46.3 ms | pNN50: 0.4% | LF/HF: 0.07 | BR: 0.2 /min
HR: 78.1 BPM | SQI: 0.84
  SDNN: 39.1 ms | RMSSD: 46.3 ms | pNN50: 0.4% | LF/HF: 0.07 | BR:

Exception in thread Thread-17:
Traceback (most recent call last):
  File "c:\Users\jisci\miniconda3\envs\ai\lib\threading.py", line 950, in _bootstrap_inner
    self.run()
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\threading.py", line 888, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 723, in <lambda>
    self.run = threading.Thread(target=lambda:self.__process_video_capture(vid_path, api))
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 841, in __process_video_capture
    self.update_frame(img, ts)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 535, in __exit__
    self.wait_completion()
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 737, in wait_completion
    

KeyboardInterrupt: 